In [1]:
import os, sys
from tqdm import tqdm
import torch
import numpy as np

sys.path.insert(0, os.path.join(os.path.abspath(''), '..'))
from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG, HARTREE2KCAL
from cmm.drivers import OptimizationDriver
from cmm.misc_utils import read_xyz

import openmm.app as app
from ase.io import read

In [2]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ff_path = os.path.join(os.path.abspath(''), '../scripts/water_refit.xml')
ff = ForceFieldXML(ff_path, device=device)

In [3]:
water_cluster_pdb_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.pdb')
water_cluster_xyz_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.xyz')

In [4]:
water_cluster_pdb = app.PDBFile(water_cluster_pdb_path)
positions = [water_cluster_pdb.getPositions(True, frame=i)._value * 10.0 for i in range(water_cluster_pdb.getNumFrames())]

topologies = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]

/home/heindelj/miniforge3/envs/pycmm/lib/python3.12/site-packages/torch/nested/__init__.py:226: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


In [5]:
coords = torch.from_numpy(positions[1] / BOHR2ANG).to(device).requires_grad_(False)
box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
energies = systems[1].getEnergy(coords, box)

for key in energies:
    print(f"{key}: {energies[key] * HARTREE2KCAL} kcal/mol")

bond: 0.40315064636088754 kcal/mol
angle: 0.023446173018394804 kcal/mol
torsion: 0.0 kcal/mol
bond_bond: -0.005611845671081472 kcal/mol
bond_angle: 0.034528992453563605 kcal/mol
angle_angle: 0.0 kcal/mol
torsion_bond: 0.0 kcal/mol
torsion_angle: 0.0 kcal/mol
torsion_angle_angle: 0.0 kcal/mol
perm_elec: -25.909409713076045 kcal/mol
pol: -3.710355919667678 kcal/mol
ct_direct: -6.5340623624889105 kcal/mol
xpol: -0.6042721511840821 kcal/mol
pauli: 27.691321653439534 kcal/mol
disp: -6.185578241307645 kcal/mol
total: -14.796842768123058 kcal/mol


In [13]:
opt_driver = OptimizationDriver(systems[-1])
coords = torch.from_numpy(positions[-1] / BOHR2ANG).to(device).requires_grad_(False)
box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
coords_opt, box_opt, result = opt_driver.run(coords, box)
print(result)
print(result.fun * HARTREE2KCAL)

  message: CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH
  success: True
   status: 0
      fun: -0.43864301255706767
        x: [-2.451e+00 -1.287e+00 ...  0.000e+00  1.000e+02]
      nit: 73
      jac: [ 4.737e-05 -9.705e-05 ...  0.000e+00  0.000e+00]
     nfev: 79
     njev: 79
 hess_inv: <234x234 LbfgsInvHessProduct with dtype=float64>
-275.25264611105047


In [11]:
reference_coords, reference_labels = read_xyz(water_cluster_xyz_path)
print(reference_coords[0])
print(coords_opt * BOHR2ANG)

[[-0.0672745  -1.52918578  0.        ]
 [ 0.06805583 -0.56628036  0.        ]
 [ 0.82629909 -1.89520487  0.        ]
 [ 0.06336087  1.39379087  0.        ]
 [-0.42571602  1.73043594  0.76340222]
 [-0.42571602  1.73043594 -0.76340222]]
tensor([[-7.2853e-02, -1.5006e+00, -3.8621e-19],
        [ 5.5045e-02, -5.4099e-01, -1.5129e-18],
        [ 8.1648e-01, -1.8560e+00,  5.8065e-18],
        [ 8.9401e-02,  1.3969e+00,  7.0075e-17],
        [-4.2504e-01,  1.6824e+00,  7.5887e-01],
        [-4.2504e-01,  1.6824e+00, -7.5887e-01]])
